# 1er Parcial - Inteligencia Artificial

Marcelo Barua
---

## 1. Objetivo 

Cada fila del archivo datos_concatenados representa los puntajes que 1 alumno consiguio en 1 asignatura en un especifico semestre y ciclo.
Cada fila de la tabla generada es un intento de un alumno, en la materia mecánica de materiales.

La variable objetivo es la columna `Aprobado`, que vale `S` o `N`, convertida a
1 y 0. Es un modelo de clasificación binaria, y el modelo devuelve además una probabilidad de aprobar.

El momento a predecir: en la inscripción, antes de que empiece el semestre, viendo las materias en las que el alumno está inscrito, y de tal forma no utilizamos el rendimiento del alumno durante el semestre. es puramente antes del inicio del semestre académico.

Las variables: el rendimiento del alumno en
asignaturas de semestres anteriores, cuántas veces ya intentó esta materia, y datos
de la inscripción (carrera, semestre, requisito).

El uso: Los alumnos para ver la probabilidad de aprobar de un compañero, los docentes para hacer acompañamiento académico a los alumnos en la materia de mecánica de materiales 1 (más específicamente en civil, ya que ahí tiene muchas correlativas)

Tenemos 1242 intentos de 2025-1 y 2025-2, ya que para los intentos en 2024-2, no tenemos datos de los previos semestres para ver. 
---

### Configuración

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid")

RUTA_CSV = "datos_concatenados.csv"
SEMILLA = 42

# Asignatura a predecir 
OBJETIVO = "MECANICA DE MATERIALES 1"

# La columna Cod.Curso indica el semestre de la malla al que pertenece cada asignatura

In [ ]:
df = pd.read_csv(RUTA_CSV, low_memory=False)

# periodo: un número que ordena los semestres. 2024 ciclo 2 -> 20242
df["periodo"] = df["Anho"] * 10 + df["Semestre"]

# variable objetivo en formato numérico
df["aprobo"] = (df["Aprobado"] == "S").astype(int)

# marca de abandono: firma en 0 significa que el alumno no llegó a cerrar el semestre
df["sin_firma"] = (df["Firma"] <= 0).astype(int)

# Semestre de malla de la materia objetivo
CURSO_OBJETIVO = int(df.loc[df["Asignatura"] == OBJETIVO, "Cod.Curso"].mode()[0])

print("Filas y columnas:", df.shape)
print("Periodos disponibles:", sorted(df["periodo"].unique()))
print(f"{OBJETIVO} pertenece al semestre {CURSO_OBJETIVO} de la malla")
df.head()

In [ ]:
# Cómo se reparten las asignaturas por semestre de malla
resumen_curso = df.groupby("Cod.Curso")["aprobo"].agg(
    inscripciones="size", asignaturas=lambda s: 0, tasa="mean")
resumen_curso["asignaturas"] = df.groupby("Cod.Curso")["Asignatura"].nunique()
print(resumen_curso.round(3))
print()
print(f"Asignaturas de los semestres 1 a {CURSO_OBJETIVO - 1}:")
for c in range(1, CURSO_OBJETIVO):
    ms = sorted(df.loc[df["Cod.Curso"] == c, "Asignatura"].unique())
    print(f"  Semestre {c} ({len(ms)}): " + ", ".join(ms[:8]) +
          (" ..." if len(ms) > 8 else ""))

## 2. Análisis de datos (20 puntos)

### 2.1 El dataset completo

In [ ]:
print("Dimensiones:", df.shape)
print()
print(df.dtypes)

In [ ]:
faltantes = df.isna().sum()
faltantes = faltantes[faltantes > 0].sort_values(ascending=False)
print("Columnas con valores faltantes:")
print(faltantes)
print()
print("Porcentaje sobre el total:")
print((faltantes / len(df) * 100).round(2))

In [ ]:
df.describe().T.round(2)

In [ ]:
# Cada fila es una inscripción, no un alumno.
print("Filas (inscripciones):", len(df))
print("Alumnos distintos:    ", df["ALUMNO_ID"].nunique())
print("Asignaturas distintas:", df["Asignatura"].nunique())
print("Filas por alumno:     ", round(len(df) / df["ALUMNO_ID"].nunique(), 2))

### 2.2 La asignatura objetivo

In [ ]:
mm = df[df["Asignatura"] == OBJETIVO]

print("Asignatura:", OBJETIVO)
print("Intentos totales:", len(mm))
print("Alumnos distintos:", mm["ALUMNO_ID"].nunique())
print("Tasa de aprobación:", round(mm["aprobo"].mean(), 4))
print()
print("Por periodo:")
print(mm.groupby("periodo")["aprobo"].agg(
    intentos="size", tasa=lambda s: round(s.mean(), 3)))
print()
print("Por carrera:")
print(mm.groupby("Carrera")["aprobo"].agg(
    intentos="size", tasa=lambda s: round(s.mean(), 3)).sort_values("tasa"))

In [ ]:
# Gráfico 1: qué tan difícil es la materia comparada con las demás
top = df["Asignatura"].value_counts().head(20).index
comp = (df[df["Asignatura"].isin(top)]
        .groupby("Asignatura")["aprobo"].mean()
        .sort_values())

colores = ["#c0392b" if a == OBJETIVO else "#7f8c8d" for a in comp.index]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(comp.index, comp.values, color=colores)
ax.axvline(df["aprobo"].mean(), color="#2c3e50", linestyle="--",
           label=f"Promedio general ({df['aprobo'].mean():.1%})")
ax.set_xlabel("Tasa de aprobación")
ax.set_title("Tasa de aprobación: 20 asignaturas con más inscripciones")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 2: cómo se reparte el resultado en la materia objetivo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

conteo = mm["aprobo"].value_counts().sort_index()
axes[0].bar(["No aprobó", "Aprobó"], conteo.values, color=["#c0392b", "#27ae60"])
for i, v in enumerate(conteo.values):
    axes[0].text(i, v, f"{v}\n({v/len(mm):.1%})", ha="center", va="bottom")
axes[0].set_title(f"Resultado en {OBJETIVO}")
axes[0].set_ylim(0, conteo.max() * 1.2)

porcar = mm.groupby("Carrera")["aprobo"].agg(["mean", "size"]).sort_values("mean")
axes[1].barh(porcar.index, porcar["mean"], color="#2c7fb8")
for i, (m, n) in enumerate(zip(porcar["mean"], porcar["size"])):
    axes[1].text(m, i, f"  n={n}", va="center")
axes[1].set_title("Tasa de aprobación por carrera")
axes[1].set_xlabel("Tasa de aprobación")

plt.tight_layout()
plt.show()

### Interpretación
La materia en el ranking de dificultad del grafico 1, esta al ultimo lugar: menos de 0.2 en tasa de aprobación
Variable objetivo: 
 - Aprobaron: 204 (16.4%)
 - No aprobaron: 1038
 - Entre carreras, la tasa varía ligeramente. aproximadamente 0.05 de variacion entre carreras, con ing civil llevando la mayor tasa de aprobacion.

# Variables: objetivo y  predictoras

La variable objetivo es 'aprobo', una variable binaria que indica si el alumno aprueba MECANICA DE MATERIALES 1 (1) o no la aprueba (0).

Las variables predictoras incluyen 30 variables numéricas construidas a partir del historial, por ejemplo: hist_tasa_aprobacion, hist_firma_media, hist_prop_sin_firma, ult_tasa_aprobacion, intentos_previos, firma_previa_objetivo, prev_tasa, max_curso_aprobado, carga_materias, carga_dificultad_media y carga_superiores. También se incluyen las variables categóricas Carrera, Semestre y Requisito.

## 3. Procesamiento de datos (20 puntos)


In [ ]:
primer_periodo = df["periodo"].min()

objetivo = df[(df["Asignatura"] == OBJETIVO) &
              (df["periodo"] > primer_periodo)].copy()
objetivo = objetivo.reset_index(drop=True)
objetivo["fila_id"] = objetivo.index

print("Intentos totales de la materia:", (df["Asignatura"] == OBJETIVO).sum())
print("Descartados por no tener semestre previo:",
      ((df["Asignatura"] == OBJETIVO) & (df["periodo"] == primer_periodo)).sum())
print("Candidatos:", len(objetivo))

In [ ]:
# Paso 2: historial anterior de cada alumno
# El merge cruza cada intento con todas las filas del mismo alumno, y después se filtra por periodo anterior
historial = objetivo[["fila_id", "ALUMNO_ID", "periodo"]].merge(
    df, on="ALUMNO_ID", suffixes=("_obj", ""))

historial = historial[historial["periodo"] < historial["periodo_obj"]]

# Se separa el historial en dos: otras materias, y los intentos previos de esta misma
otras = historial[historial["Asignatura"] != OBJETIVO]
mismas = historial[historial["Asignatura"] == OBJETIVO]

print("Filas de historial:", len(historial))
print("  en otras materias:", len(otras))
print("  en la materia objetivo:", len(mismas))
print()
print("Materias previas por intento (mediana):",
      otras.groupby("fila_id").size().median())

In [ ]:
# Paso 3: resumir el historial en columnas numéricas
g = otras.groupby("fila_id")

rasgos = pd.DataFrame({
    # volumen de experiencia
    "hist_inscripciones":     g.size(),
    "hist_asignaturas":       g["Asignatura"].nunique(),
    "hist_periodos":          g["periodo"].nunique(),
    # rendimiento
    "hist_aprobadas":         g["aprobo"].sum(),
    "hist_tasa_aprobacion":   g["aprobo"].mean(),
    "hist_firma_media":       g["Firma"].mean(),
    "hist_firma_max":         g["Firma"].max(),
    "hist_primer_par_media":  g["Primer.Par"].mean(),
    "hist_segundo_par_media": g["Segundo.Par"].mean(),
    "hist_asis_media":        g["Asis"].mean(),
    # abandono
    "hist_prop_sin_firma":    g["sin_firma"].mean(),
})

print("Rasgos de historial general:", rasgos.shape)
rasgos.head()

In [ ]:
# Rendimiento en el ÚLTIMO semestre cursado: mide el momento actual del alumno, no su promedio histórico
ultimo = otras.groupby("fila_id")["periodo"].max().rename("ult")
recientes = otras.merge(ultimo, on="fila_id")
recientes = recientes[recientes["periodo"] == recientes["ult"]].groupby("fila_id")

rasgos["ult_tasa_aprobacion"] = recientes["aprobo"].mean()
rasgos["ult_firma_media"]     = recientes["Firma"].mean()
rasgos["ult_inscripciones"]   = recientes.size()

# Intentos anteriores de la misma materia
gm = mismas.groupby("fila_id")
rasgos["intentos_previos"]      = gm.size()
rasgos["firma_previa_objetivo"] = gm["Firma"].max()

# Rendimiento en las materias de semestres ANTERIORES al de la materia objetivo
# Cod.Curso < CURSO_OBJETIVO selecciona esas asignaturas sin lista escrita a mano
previas = otras[otras["Cod.Curso"] < CURSO_OBJETIVO].groupby("fila_id")
rasgos["prev_cursadas"]    = previas.size()
rasgos["prev_aprobadas"]   = previas["aprobo"].sum()
rasgos["prev_tasa"]        = previas["aprobo"].mean()
rasgos["prev_firma_media"] = previas["Firma"].mean()

# Dónde está parado el alumno dentro de la malla
rasgos["curso_medio_previo"]  = otras.groupby("fila_id")["Cod.Curso"].mean()
rasgos["max_curso_aprobado"]  = (otras[otras["aprobo"] == 1]
                                 .groupby("fila_id")["Cod.Curso"].max())
rasgos["prop_curso_superior"] = (otras.assign(
    sup=(otras["Cod.Curso"] >= CURSO_OBJETIVO).astype(int))
    .groupby("fila_id")["sup"].mean())

print("Rasgos hasta acá:", rasgos.shape[1])
list(rasgos.columns)

### Carga del semestre

Cursar la materia objetivo sola no es lo mismo que cursarla junto con otras cinco.
Cuántas materias anota el alumno se sabe en la inscripción, así que es una
variable válida.

También importa cuáles: acompañarla de asignaturas fáciles no es lo mismo que
acompañarla de asignaturas difíciles.

Sin embargo, como la dificultad de una materia es su tasa de aprobación, 
y esta tasa se calcula con resultados, si usamos los resultados
del mismo semestre usaríamos información del futuro. 
Por eso las dificultades de las materias se calculan usando solo los periodos anteriores.

In [ ]:
# Dificultad de cada asignatura, calculada por separado para cada periodo objetivo
# usando únicamente datos anteriores a ese periodo
dificultad = {}
for p in sorted(objetivo["periodo"].unique()):
    anterior = df[df["periodo"] < p]
    dificultad[p] = anterior.groupby("Asignatura")["aprobo"].mean()
    print(f"Periodo {p}: dificultad calculada con {len(anterior)} filas anteriores,",
          f"{len(dificultad[p])} asignaturas")

In [ ]:
# Materias que el alumno cursa en el MISMO periodo que la materia objetivo
simultaneas = objetivo[["fila_id", "ALUMNO_ID", "periodo"]].merge(
    df[["ALUMNO_ID", "periodo", "Asignatura", "Cod.Curso"]], on=["ALUMNO_ID", "periodo"])

rasgos = rasgos.join(simultaneas.groupby("fila_id").size().rename("carga_materias"))

# Solo las OTRAS materias: la dificultad de la objetivo es igual para todos
acompanantes = simultaneas[simultaneas["Asignatura"] != OBJETIVO].copy()
acompanantes["dif"] = [dificultad[p].get(a, np.nan) for p, a
                       in zip(acompanantes["periodo"], acompanantes["Asignatura"])]

ga = acompanantes.groupby("fila_id")
rasgos["carga_otras"]            = ga.size()
rasgos["carga_dificultad_media"] = ga["dif"].mean()
rasgos["carga_dificultad_min"]   = ga["dif"].min()   # la materia más difícil del semestre

# Nivel de malla de las materias que acompañan
rasgos["carga_curso_medio"] = ga["Cod.Curso"].mean()
rasgos["carga_curso_max"]   = ga["Cod.Curso"].max()
rasgos["carga_superiores"]  = (acompanantes.assign(
    sup=(acompanantes["Cod.Curso"] >= CURSO_OBJETIVO).astype(int))
    .groupby("fila_id")["sup"].sum())

print("Total de rasgos construidos:", rasgos.shape[1])
list(rasgos.columns)

In [ ]:
# Unir los rasgos con las filas objetivo
datos = objetivo.merge(rasgos, on="fila_id", how="inner")

# Sin historial no hay nada que predecir
datos = datos[datos["hist_inscripciones"] > 0].copy()

# Estas cuentas quedan vacías cuando el alumno nunca cursó eso: vacío significa cero
for c in ["intentos_previos", "firma_previa_objetivo",
          "prev_cursadas", "prev_aprobadas", "carga_otras", "carga_superiores"]:
    datos[c] = datos[c].fillna(0)
datos["carga_materias"] = datos["carga_materias"].fillna(1)

print("Tabla final:", datos.shape)
print("Aprobaron:", int(datos["aprobo"].sum()),
      f"({datos['aprobo'].mean():.1%})")
print("No aprobaron:", int((1 - datos["aprobo"]).sum()))
datos[list(rasgos.columns)].head()

In [ ]:
# Cobertura: cuántas filas tienen realmente cada rasgo nuevo
nuevos = ["prev_cursadas", "prev_tasa", "max_curso_aprobado", "prop_curso_superior",
          "carga_materias", "carga_dificultad_media", "carga_curso_max", "carga_superiores"]
for c in nuevos:
    con_dato = datos[c].notna().sum()
    print(f"{c:24s} con valor: {con_dato:5d} de {len(datos)} "
          f"({con_dato/len(datos):.0%})")

print()
print(f"Materias de semestres 1 a {CURSO_OBJETIVO - 1} en el historial disponible:")
print(datos["prev_cursadas"].value_counts().sort_index())

In [ ]:
# Qué tan relacionado está cada rasgo con el resultado
correlaciones = (datos[list(rasgos.columns) + ["aprobo"]]
                 .corr()["aprobo"].drop("aprobo")
                 .sort_values(key=abs, ascending=False)
                 .round(3))
print("Correlación de cada rasgo con aprobar:")
print(correlaciones.to_string())

### 3.2 Análisis de la tabla construida


In [ ]:
datos[list(rasgos.columns)].describe().T.round(2)

In [ ]:
# Gráfico 3: historial académico según el resultado
variables = ["hist_tasa_aprobacion", "hist_firma_media",
             "hist_prop_sin_firma", "ult_tasa_aprobacion"]

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, col in zip(axes, variables):
    sns.boxplot(data=datos, x="aprobo", y=col, ax=ax,
                hue="aprobo", palette=["#c0392b", "#27ae60"], legend=False)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["No aprobó", "Aprobó"])
fig.suptitle("Historial previo según el resultado en la materia objetivo")
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 4: tasa de aprobación según intentos previos y según requisito
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

por_intento = datos.groupby("intentos_previos")["aprobo"].agg(["mean", "size"])
axes[0].bar(por_intento.index.astype(int).astype(str), por_intento["mean"],
            color="#2c7fb8")
for i, (m, n) in enumerate(zip(por_intento["mean"], por_intento["size"])):
    axes[0].text(i, m, f"n={n}", ha="center", va="bottom")
axes[0].set_title("Tasa de aprobación según intentos previos")
axes[0].set_xlabel("Veces que ya cursó la materia")
axes[0].set_ylim(0, max(por_intento["mean"]) * 1.3)

por_req = datos.groupby("Requisito")["aprobo"].agg(["mean", "size"])
axes[1].bar(por_req.index.astype(str), por_req["mean"], color="#e67e22")
for i, (m, n) in enumerate(zip(por_req["mean"], por_req["size"])):
    axes[1].text(i, m, f"n={n}", ha="center", va="bottom")
axes[1].set_title("Tasa de aprobación según Requisito")
axes[1].set_xlabel("Requisito cumplido")
axes[1].set_ylim(0, max(por_req["mean"]) * 1.3)

plt.tight_layout()
plt.show()

print(por_req.round(3))

In [ ]:
# Gráfico 5:  carga del semestre complica o ayuda
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

carga = datos.groupby("carga_materias")["aprobo"].agg(["mean", "size"])
carga = carga[carga["size"] >= 20]
axes[0].bar(carga.index.astype(int).astype(str), carga["mean"], color="#8e44ad")
for i, (m, n) in enumerate(zip(carga["mean"], carga["size"])):
    axes[0].text(i, m, f"n={int(n)}", ha="center", va="bottom", fontsize=8)
axes[0].axhline(datos["aprobo"].mean(), color="#c0392b", linestyle="--",
                label="Promedio de la materia")
axes[0].set_title("Tasa de aprobación según cantidad de materias del semestre")
axes[0].set_xlabel("Materias cursadas en el mismo periodo")
axes[0].set_ylim(0, carga["mean"].max() * 1.35)
axes[0].legend(fontsize=8)

nivel = datos.groupby("carga_curso_max")["aprobo"].agg(["mean", "size"])
nivel = nivel[nivel["size"] >= 20]
colores = ["#c0392b" if c <= CURSO_OBJETIVO else "#27ae60" for c in nivel.index]
axes[1].bar(nivel.index.astype(int).astype(str), nivel["mean"], color=colores)
for i, (m, n) in enumerate(zip(nivel["mean"], nivel["size"])):
    axes[1].text(i, m, f"n={int(n)}", ha="center", va="bottom", fontsize=8)
axes[1].set_title("Tasa según el semestre más alto que cursa en paralelo")
axes[1].set_xlabel("Cod.Curso más alto entre las materias simultáneas")
axes[1].set_ylim(0, nivel["mean"].max() * 1.3)

plt.tight_layout()
plt.show()

print("Por cantidad de materias:")
print(carga.round(3))
print()
print("Por semestre más alto cursado en paralelo:")
print(nivel.round(3))

In [ ]:
NUMERICAS = list(rasgos.columns)
CATEGORICAS = ["Carrera", "Semestre", "Requisito"]

X = datos[NUMERICAS + CATEGORICAS]
y = datos["aprobo"]

print("Variables numéricas:", len(NUMERICAS))
print("Variables categóricas:", len(CATEGORICAS))
print("X:", X.shape, " y:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

# test_size=0.25 en vez de 0.20: con pocas filas conviene un conjunto de prueba
# más grande para que las métricas no dependan de algunos casos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEMILLA, stratify=y)

print("Entrenamiento:", X_train.shape, " aprobados:", int(y_train.sum()))
print("Prueba:       ", X_test.shape,  " aprobados:", int(y_test.sum()))
print()
print("Tasa de aprobación - train:", round(y_train.mean(), 4),
      " test:", round(y_test.mean(), 4))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocesador = ColumnTransformer([
    ("num", Pipeline([
        ("imputar", SimpleImputer(strategy="median")),
        ("escalar", StandardScaler()),
    ]), NUMERICAS),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAS),
])

prueba = preprocesador.fit_transform(X_train)
print("Columnas después del preprocesamiento:", prueba.shape[1])

### Decisiones tomadas
Se construyó una tabla analítica con una fila por intento en MECANICA DE MATERIALES 1, en lugar de usar directamente el CSV, donde cada fila representa una inscripción en cualquier asignatura.

Para cada intento se usó únicamente el historial anterior del alumno (periodo < periodo_obj). Esto evita fuga de información: no se incorporan notas ni resultados que todavía no existen al momento de la inscripción. Se descartaron los 384 intentos de 2024-2 porque no tienen un semestre previo disponible.

La variable objetivo es aprobo, derivada de Aprobado: S = 1 y N = 0. La tabla final contiene 1.242 intentos, de los cuales 204 aprobaron (16,4 %).

Se imputaron los faltantes numéricos con la mediana, ya que es menos sensible a valores extremos. En variables de conteo, como intentos_previos, los faltantes se reemplazaron por cero porque representan que el alumno nunca tuvo ese antecedente.

Se usaron variables disponibles al inscribirse: historial académico, intentos previos, carga de materias, nivel de la malla, carrera, semestre y requisito. Se excluyeron, entre otras, Primer.Par, Segundo.Par, Firma y Nota.Final, porque pertenecen al semestre que se quiere predecir o registran su desenlace.

Las variables categóricas (Carrera, Semestre y Requisito) se transformaron con One-Hot Encoding. Con handle_unknown="ignore" evitamos errores si aparece una categoría nueva en prueba.

Se dividieron los datos en 75 % para entrenamiento y 25 % para prueba, con stratify=y para mantener la proporción de aprobados. La imputación y el escalado se dejaron dentro del Pipeline para evitar fuga de información desde el conjunto de prueba.

Se aplicó class_weight="balanced" en todos los modelos, porque la clase “aprobó” representa solo el 16,4 % de los casos.



## 4. Entrenamiento de modelos

Tres modelos con lógicas distintas:

1. **Regresión Logística** — modelo lineal; devuelve directamente una probabilidad. max_iter=2000, class_weight="balanced", random_state=42
2. **Árbol de Decisión** — reglas encadenadas; se puede leer y explicar. max_depth=4, min_samples_leaf=20, class_weight="balanced", random_state=42
3. **Random Forest** — promedio de 300 árboles; capta combinaciones no lineales. n_estimators=300, max_depth=8, min_samples_leaf=5, class_weight="balanced", n_jobs=-1, random_state=42

**`class_weight="balanced"`** aparece en los tres. Solo el 16% de los intentos
termina en aprobación. Sin ese parámetro, a los modelos les conviene decir
"no aprueba" siempre. Con él, cada caso de aprobación pesa más durante el
entrenamiento.

En los tres casos se aplicó imputación por mediana, escalado de variables numéricas y One-Hot Encoding para las categóricas. El escalado es especialmente importante para la regresión logística; los modelos de árboles no dependen de la escala.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

modelos = {
    "Regresión Logística": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=SEMILLA,
    ),
    "Árbol de Decisión": DecisionTreeClassifier(
        max_depth=4,            # poco profundo: con 1.200 filas se memoriza rápido
        min_samples_leaf=20,    # cada hoja necesita al menos 20 casos
        class_weight="balanced",
        random_state=SEMILLA,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=5,
        class_weight="balanced",
        n_jobs=-1,
        random_state=SEMILLA,
    ),
}

for nombre, m in modelos.items():
    print(nombre, "->", m)

In [ ]:
entrenados = {}
for nombre, modelo in modelos.items():
    pipe = Pipeline([("prep", preprocesador), ("modelo", modelo)])
    pipe.fit(X_train, y_train)
    entrenados[nombre] = pipe
    print("Entrenado:", nombre)



## 5. Pruebas y comparación

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score)
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Validación cruzada: con 1.200 filas
# 5 particiones dan una estimación más estable
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

filas = []
for nombre, pipe in entrenados.items():
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    auc_cv = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
    filas.append({
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba),
        "PR-AUC": average_precision_score(y_test, proba),
        "ROC-AUC (CV)": auc_cv.mean(),
        "± desvío CV": auc_cv.std(),
    })

comparativa = pd.DataFrame(filas).set_index("Modelo").round(3)
print("Referencia - accuracy de decir 'no aprueba' siempre:",
      round(1 - y_test.mean(), 3))
print()
comparativa

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (nombre, pipe) in zip(axes, entrenados.items()):
    ConfusionMatrixDisplay.from_estimator(
        pipe, X_test, y_test, ax=ax, cmap="Blues", colorbar=False,
        display_labels=["No aprobó", "Aprobó"])
    ax.set_title(nombre, fontsize=10)
fig.suptitle("Matrices de confusión sobre el conjunto de prueba")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for nombre, pipe in entrenados.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, ax=axes[0], name=nombre)
    PrecisionRecallDisplay.from_estimator(pipe, X_test, y_test, ax=axes[1], name=nombre)

axes[0].plot([0, 1], [0, 1], "k--", linewidth=1, label="Azar")
axes[0].set_title("Curva ROC")
axes[0].legend(fontsize=8)

axes[1].axhline(y_test.mean(), color="k", linestyle="--", linewidth=1)
axes[1].set_title("Curva Precision-Recall")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

### La probabilidad de aprobar

Esto es lo que pedía el objetivo: no una respuesta de sí o no, sino un número
entre 0 y 1 para cada alumno.

In [ ]:
mejor = comparativa["ROC-AUC"].idxmax()
print("Mejor modelo según ROC-AUC:", mejor)

probabilidades = entrenados[mejor].predict_proba(X_test)[:, 1]

ranking = pd.DataFrame({
    "prob_aprobar": probabilidades,
    "resultado_real": np.where(y_test.values == 1, "Aprobó", "No aprobó"),
    "hist_tasa_aprobacion": X_test["hist_tasa_aprobacion"].values.round(2),
    "hist_firma_media": X_test["hist_firma_media"].values.round(1),
    "intentos_previos": X_test["intentos_previos"].values.astype(int),
}).sort_values("prob_aprobar", ascending=False).round(3)

print("\n10 alumnos con MAYOR probabilidad estimada:")
print(ranking.head(10).to_string(index=False))
print("\n10 alumnos con MENOR probabilidad estimada:")
print(ranking.tail(10).to_string(index=False))

In [ ]:
# ¿Las probabilidades son confiables? Agrupamos por tramo y comparamos
# lo que el modelo predijo con lo que realmente pasó.
tramos = pd.cut(probabilidades, [0, .1, .25, .5, .75, 1.0])
calib = pd.DataFrame({"tramo": tramos, "real": y_test.values}).groupby(
    "tramo", observed=True)["real"].agg(alumnos="size", tasa_real="mean").round(3)

print("Probabilidad estimada vs resultado real:")
print(calib)

In [ ]:
from sklearn.metrics import precision_score, recall_score

print(f"{'umbral':>8} {'marcados':>10} {'precision':>10} {'recall':>8}")
for u in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    pred_u = (probabilidades >= u).astype(int)
    print(f"{u:>8} {pred_u.sum():>10} "
          f"{precision_score(y_test, pred_u, zero_division=0):>10.3f} "
          f"{recall_score(y_test, pred_u):>8.3f}")

El mejor modelo es Random Forest. Se seleccionó por su ROC-AUC de 0,943 en prueba y 0,950 ± 0,030 en validación cruzada, además de alcanzar los mejores valores de F1, recall y PR-AUC. ROC-AUC es más útil que accuracy porque el conjunto está desbalanceado: un clasificador que siempre responde “no aprueba” lograría 83,6 % de accuracy, pero no detectaría ningún aprobado. Random Forest alcanza 88,7 % de accuracy y, sobre todo, ordena mucho mejor a los alumnos según su probabilidad de aprobación. La diferencia frente a los otros modelos es mayor que el desvío de la validación cruzada, por lo que la elección es razonablemente consistente.

En su matriz de confusión, Random Forest obtuvo 229 verdaderos negativos, 47 verdaderos positivos, 31 falsos positivos y 4 falsos negativos. Para un programa de tutorías conviene tolerar más falsos negativos desde la perspectiva de 'aprobación': es decir, ofrecer apoyo a alguien que finalmente aprobaría. antes que falsos positivos, que equivalen a clasificar como seguro a un alumno que finalmente no aprueba. Para una capacidad fija de 40 tutorías, se debe seleccionar a los 40 alumnos con menor prob_aprobar, en lugar de usar un umbral fijo de 0,5.

---

## 6. Conclusiones 

MECANICA DE MATERIALES 1 es una asignatura difícil: en la tabla final solo aprueba el 16,4 % de los intentos. Las variables más asociadas con aprobar son el desempeño previo en la misma materia (firma_previa_objetivo), la cantidad de intentos anteriores, el promedio histórico de firma y la tasa histórica de aprobación. Por lo tanto, el rendimiento académico previo aporta información útil para estimar la probabilidad de aprobar antes de iniciar el semestre.

La carga del semestre no confirmó la hipótesis simple de que cursar más materias reduce la aprobación. Su correlación fue débil y positiva (0,081), posiblemente porque los alumnos que se inscriben en más asignaturas también son quienes tienen mejor trayectoria académica. Además con Cod.Curso pudimos representar el semestre de la malla y construir variables de avance académico sin definir listas manuales de asignaturas.

El análisis tiene como principal limitación que el CSV solo contiene tres periodos académicos. Esto restringe el historial disponible, especialmente para los alumnos de periodos iniciales. Como mejoras futuras, se pueden incorporar más semestres, ajustar hiperparámetros con GridSearchCV, probar modelos como XGBoost y agregar variables temporales, por ejemplo los semestres transcurridos desde la aprobación de materias base, si el estudiante trabaja o no, etc.